## torch.multinomial 是 PyTorch 里按概率采样的函数：
“给定一个概率向量，按这些概率有放回地抽若干个样本”——语言模型生成 token、强化学习选动作都离不开它。

torch.multinomial(probs, num_samples, replacement=False, *, generator=None)
- probs：非负权重或概率，最后一维求和必须 > 0
- num_samples：要抽几个样本
- replacement：是否有放回（默认 False）
- 返回：LongTensor，里面是采样下标

In [3]:
import torch
probs = torch.tensor([0.1, 0.3, 0.5, 0.1])   # 不必归一 1，函数内部会除和
idx = torch.multinomial(probs, 1)              # 抽 1 个
print(idx.item())                              # 大概率输出 2

2


运行 3 次可能看到：2, 1, 2 —— 0.5 的类别被抽得最频繁。

## 等价于 logits.argmax(-1)的随机版
- 温度、top-k、top_p 只是提前把 probs 截断或平滑。

In [ ]:
logits = model(**inputs).logits[:, -1, :]          # [batch, vocab]
probs  = logits.softmax(-1)                        # 变概率
next_id = torch.multinomial(probs, num_samples=1)  # [batch, 1]

## 强化学习选动作

In [ ]:
action_probs = policy_net(state)                   # [N_actions]
action = torch.multinomial(action_probs, 1).item() # 0~N-1 整数
logp   = action_probs.log_softmax(-1)[action]      # 留好 log_prob 供回放

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("gpt2")
tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained("gpt2")

prompt = "The answer is"
inputs = tok(prompt, return_tensors='pt')

# ① 取最后一个位置的 logits
with torch.no_grad():
    logits = model(**inputs).logits[:, :, :]   # ← 关键一行
    print('logits_all:',logits)
    logits = logits[:, -1, :]
    print('logits_last:',logits)
print("logits shape:", logits.shape)            # [1, 50257]

# ② 变成概率并采样
probs = logits.softmax(-1)
next_id = torch.multinomial(probs, 1)           # [1, 1]
next_token = tok.decode(next_id.squeeze().tolist())
# next_token = tok.decode(next_id, skip_special_tokens=True)

# ③ 打印结果
print("Top-5 概率词：")
top5 = torch.topk(probs[0], 5)
for i, (p, idx) in enumerate(zip(top5.values, top5.indices)):
    print(f"{i+1}. {tok.decode([idx])!r:>10}  {p.item():.3%}")

print("\n采样结果：", repr(next_token))

In [ ]:
logits_all: tensor([[[-36.2874, -35.0114, -38.0793,  ..., -40.5163, -41.3759, -34.9193],
         [-68.6779, -70.5239, -75.9774,  ..., -79.6497, -78.8252, -71.9340],
         [-90.2095, -90.9617, -95.9635,  ..., -98.3604, -99.0060, -95.0042]]])
logits_last: tensor([[-90.2095, -90.9617, -95.9635,  ..., -98.3604, -99.0060, -95.0042]])
logits shape: torch.Size([1, 50257])
Top-5 概率词：
1.    ' that'  13.285%
2.  ' simple'  9.045%
3.        ','  6.825%
4.      ' no'  5.164%
5.        ':'  5.039%

采样结果： ' that'